<a href="https://colab.research.google.com/github/tazza11/brigadoon/blob/main/notebooks/link01_decision_model_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LINK-01 로봇 의사결정 모델 만들기 (Colab)

**CONNECT AI LAB × AI CITY BUILDERS** · [CONNECT AI WORLD 로봇 웹 학습기](https://www.ezerai.xyz/robot-lab/world/) · [CONNECT TYCOON](https://github.com/wonseokjung/connect-tycoon)

웹 학습기에서 내가 코스(장애물 · 목표 · 출발점)를 만들고 브라우저 안에서 PPO로 키운 **LINK-01 결정 두뇌(brain.json)** 를,
언어 모델에 옮겨 담아 **로봇 의사결정 AI 모델**로 만들고 Hugging Face에 올립니다.

1. **두뇌 불러오기** · 내 brain.json + 코스 파일 world.json (없으면 예시 두뇌 · 「장애물 피하기」)
2. **문제집** · 두뇌가 코스를 걷는 동안 0.5초마다, 두뇌가 본 숫자(목표 거리 · 방향 · 거리 센서 · 속도)를 글로 바꾼 문제 · 정답 = 두뇌의 답
3. **학습 전** · 아무것도 배우지 않은 언어 모델을 로봇의 결정 자리에 앉혀 봅니다
4. **지도학습** · 언어 모델이 내 두뇌의 답을 그대로 배웁니다 (LoRA)
5. **강화학습** · 두뇌의 답보다 확실히 나은 보기는 올리고, 확실히 나쁜 보기는 내립니다
6. **공개** · 같은 코스 20판 성공률이 더 높은 쪽을 내 Hugging Face에 `connectailab-rl-이름-버전-link01`

> 메뉴 **런타임 → 런타임 유형 변경 → GPU (T4 이상)**. 왼쪽 🔑(보안 비밀)에 `HF_TOKEN`(Write 권한)을 넣어 두면 마지막 셀에서 바로 올라갑니다.

**걷기에 대해** · 웹에서는 두뇌가 고른 방향이 미리 배운 걷기 신경망 → MuJoCo 물리로 걷습니다. 이 노트북은 로봇 모델 없이, 그 걷기를 실제로 재서 만든 **2D 걷기 근사**(명령마다 실제 속도 · 회전 · 흔들림)로 걷습니다. 넘어짐이 없어서 웹보다 성공률이 조금 후할 수 있습니다.

In [1]:
# 설치 (2~3분)
!pip -q install "connect-tycoon @ git+https://github.com/wonseokjung/connect-tycoon" "transformers==5.18.0" peft accelerate huggingface_hub
!pip -q install flash-linear-attention || echo "(빠른 계산 모듈 없이 진행 · 조금 느릴 뿐 결과는 같습니다)"
!pip -q uninstall -y torchao   # Colab 기본 torchao(0.10)가 최신 peft와 맞지 않습니다 · 이 학습에는 필요 없습니다
import torch, connect_tycoon as ct
print("CONNECT TYCOON", ct.__version__, "· GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임을 GPU로 바꾸세요)")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
CONNECT TYCOON 0.1.0 · GPU Tesla T4


## 1 · 내 LINK-01 두뇌 불러오기

[로봇 웹 학습기](https://www.ezerai.xyz/robot-lab/world/)에서 학습을 마치고 **brain.json** 칸의 **받기**로 받은 파일(`link01-과제이름-mine-brain.json`)과, **world.json** 칸의 **코스 저장**으로 받은 `…world.json`입니다.
- **끌어다 놓기** · Colab 왼쪽 **📁(파일)** 창에 두 파일을 같이 끌어다 놓고 실행 (가장 확실)
- **파일 선택 버튼** · 파일 창에 없으면 이 셀 아래에 [파일 선택] 버튼이 뜹니다 (두 파일을 같이 골라도 됩니다)
- world.json이 없으면 그 과제의 **무작위 코스**(웹 공식 시험과 같은 코스들)로 진행합니다
- `USE_MY_BRAIN`을 끄면 예시 두뇌(「장애물 피하기」 · 내 코스 하나에서 20갱신)로 해 봅니다

**왜** · 두뇌가 배운 그 코스에서 문제를 만들고, 그 코스에서 의사결정 모델을 걷게 해야 비교가 됩니다.

✅ **정상이면** · 코스 지문이 두뇌의 trained_on에 있고, 2D 걷기로 다시 걸어 본 두뇌 성공률이 웹의 「이 코스 20판」 근처입니다 (예시 두뇌 100%)

In [2]:
USE_MY_BRAIN = True                                    #@param {type:"boolean"}
UPLOAD_BUTTON = True                                   #@param {type:"boolean"}
import os
from connect_tycoon.link01 import sim, questions as Q
from connect_tycoon.link01.brain import WorldBrain, example, is_world

if USE_MY_BRAIN:
    from connect_tycoon.colab import find_brain, find_course   # 📁에 있으면 그것, 없으면 [파일 선택] 버튼
    path = find_brain(upload_button=UPLOAD_BUTTON)
    assert is_world(path), "LINK-01 로봇 두뇌(connect-world-brain/1)가 아닙니다 · 게임 두뇌는 Colab ①, 물류 두뇌는 Colab ③을 쓰세요"
    brain, WORLD = WorldBrain.from_file(path), find_course(os.path.dirname(os.path.abspath(path)))
else:
    brain, WORLD = example()
COURSE = None
if WORLD:
    task, COURSE, HASH = sim.load_course(WORLD)
    assert task == brain.task, f"코스는 「{sim.TASK_KO[task]}」, 두뇌는 「{sim.TASK_KO[brain.task]}」입니다 · 같은 과제의 world.json을 넣어 주세요"
    print(f"코스 {os.path.basename(WORLD)} · 지문 {HASH}", "· 이 두뇌가 배운 코스 ✓" if HASH in brain.trained_on else "· ⚠ 이 두뇌가 배운 코스가 아닙니다")
else:
    print("world.json 없음 · 그 과제의 무작위 코스(씨앗별로 다른 코스)로 진행합니다")
mine = sim.evaluate(brain.act, brain.task, COURSE, rays=brain.rays)
print(f"두뇌 {brain.name} · {sim.TASK_KO[brain.task]} · {brain.contract}")
print(f"  웹 공식 시험(무작위 코스 20판) {brain.test.get('score', 0):.0%} · 2D 걷기로 {'이 코스' if COURSE else '무작위 코스'} 20판 {mine['success']:.0%} (부딪힌 판 {mine['collide']:.0%})")

아래 [파일 선택] 버튼을 눌러 brain.json을 고르세요.
버튼이 안 보이거나 눌러도 반응이 없으면: 이 셀을 멈추고(■) → 왼쪽 📁(파일) 창에 brain.json을 끌어다 놓고 → 이 셀을 다시 실행(▶)


Saving link01-avoid_obstacles-py-brain.json to link01-avoid_obstacles-py-brain.json
찾은 brain.json · link01-avoid_obstacles-py-brain.json
world.json 없음 · 그 과제의 무작위 코스(씨앗별로 다른 코스)로 진행합니다
두뇌 강화학습 AI · 장애물 피하기 · link01-decision/avoid_obstacles/obs21-act5/walker51-14/v1
  웹 공식 시험(무작위 코스 20판) 90% · 2D 걷기로 무작위 코스 20판 95% (부딪힌 판 10%)


In [3]:
import json

# 현재 인식된 두뇌 파일 경로
brain_file_path = 'link01-avoid_obstacles-py-brain.json'

with open(brain_file_path, 'r', encoding='utf-8') as f:
    brain_data = json.load(f)

# 두뇌가 학습한 코스의 고유 해시값(지문) 출력
trained_courses = brain_data.get('trained_on', [])
print("=== [두뇌 학습 코스 정보] ===")
print(f"이 두뇌가 학습한 코스 고유 해시 목록: {trained_courses}")
print("\n※ 업로드하신 world.json 파일명에 포함된 8자리 해시(예: 3dde73b1)가 위 목록에 들어있어야 정상 매칭됩니다!")

=== [두뇌 학습 코스 정보] ===
이 두뇌가 학습한 코스 고유 해시 목록: ['random']

※ 업로드하신 world.json 파일명에 포함된 8자리 해시(예: 3dde73b1)가 위 목록에 들어있어야 정상 매칭됩니다!


In [4]:
import os
print("현재 업로드된 파일 목록:")
for f in sorted(os.listdir('.')):
    if os.path.isfile(f):
        print(f"- {f}")

현재 업로드된 파일 목록:
- link01-avoid_obstacles-py-brain.json


In [9]:
import os
print("=== 현재 실제 저장된 파일 목록 ===")
files_list = [f for f in os.listdir('.') if os.path.isfile(f)]
if not files_list:
    print("현재 업로드된 파일이 없습니다.")
else:
    for f in sorted(files_list):
        print(f"- {f}")

=== 현재 실제 저장된 파일 목록 ===
- link01-avoid_obstacles-3dde73b1.world.json


In [5]:
import os
from connect_tycoon.link01.brain import WorldBrain
from connect_tycoon.colab import find_brain

# 중복 업로드로 인해 이름 뒤에 (1) 등이 붙은 경우 원본 파일명으로 덮어쓰기 정리
for f in os.listdir('.'):
    if 'avoid_obstacles' in f and ('(' in f or ' ' in f):
        target = 'link01-avoid_obstacles-py-brain.json'
        if os.path.exists(target):
            os.remove(target)
        os.rename(f, target)
        print(f"✓ 중복된 파일 정렬 및 변경 완료: {f} -> {target}")

print("=== [1] 현재 폴더의 주요 파일 목록 ===")
files_list = [f for f in os.listdir('.') if os.path.isfile(f) and f.endswith('.json')]
for f in sorted(files_list):
    print(f"- {f}")

try:
    brain_path = find_brain(upload_button=False)
    test_brain = WorldBrain.from_file(brain_path)
    print("\n=== [2] 업로드된 두뇌 검증 결과 ===")
    print(f"✓ 파일명: {os.path.basename(brain_path)}")
    print(f"✓ 과제 종류: {test_brain.task}")
    if test_brain.task == 'avoid_obstacles':
        print("\n🎉 [검증 통과] '장애물 피하기' 두뇌가 맞습니다! 이제 '1 · 내 LINK-01 두뇌 불러오기' 셀(seJscKepto3j)을 실행해 주세요.")
    else:
        print("\n⚠ [검증 실패] 파일이 여전히 다른 과제 두뇌입니다. 올바른 파일을 다시 업로드해 주세요.")
except Exception as e:
    print(f"\n⚠ [파일 에러] 두뇌 파일 인식 실패: {e}")

=== [1] 현재 폴더의 주요 파일 목록 ===
- link01-avoid_obstacles-py-brain.json
찾은 brain.json · link01-avoid_obstacles-py-brain.json

=== [2] 업로드된 두뇌 검증 결과 ===
✓ 파일명: link01-avoid_obstacles-py-brain.json
✓ 과제 종류: avoid_obstacles

🎉 [검증 통과] '장애물 피하기' 두뇌가 맞습니다! 이제 '1 · 내 LINK-01 두뇌 불러오기' 셀(seJscKepto3j)을 실행해 주세요.


In [6]:
import os
from connect_tycoon.link01.brain import WorldBrain
from connect_tycoon.colab import find_brain

try:
    brain_path = find_brain(upload_button=False)
    test_brain = WorldBrain.from_file(brain_path)
    print(f"✓ 인식된 두뇌 파일명: {os.path.basename(brain_path)}")
    print(f"✓ 두뇌에 등록된 과제: {test_brain.task} ({'장애물 피하기가 맞습니다!' if test_brain.task == 'avoid_obstacles' else '⚠ 아직 물건 중 고르기 두뇌입니다. 파일을 다시 업로드해주세요.'})")
except Exception as e:
    print(f"⚠ 에러 발생: {e}\n두뇌 파일(.json)이 올바르게 업로드되었는지 확인해 주세요.")

찾은 brain.json · link01-avoid_obstacles-py-brain.json
✓ 인식된 두뇌 파일명: link01-avoid_obstacles-py-brain.json
✓ 두뇌에 등록된 과제: avoid_obstacles (장애물 피하기가 맞습니다!)


In [7]:
import os
from google.colab import files

# 1. 경로 따라가기 등 잘못 업로드된 기존 두뇌 파일 일괄 정리
wrong_brains = ['link01-choose_object-mine-brain.json', 'link01-follow_route-py-brain.json']
for wb in wrong_brains:
    if os.path.exists(wb):
        os.remove(wb)
        print(f"✓ 잘못 구성된 파일 삭제 완료: {wb}")

# 2. 장애물 피하기 두뇌 파일 업로드
print("\n[업로드 안내] 다운로드 받으신 '장애물 피하기' 두뇌 파일(.json)을 선택해 주세요.")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"✓ 새 파일 업로드 완료: {filename}")


[업로드 안내] 다운로드 받으신 '장애물 피하기' 두뇌 파일(.json)을 선택해 주세요.


Saving link01-avoid_obstacles-py-brain.json to link01-avoid_obstacles-py-brain (1).json
✓ 새 파일 업로드 완료: link01-avoid_obstacles-py-brain (1).json


## 2 · 문제집 · 두뇌의 결정을 글 문제로 (약 3~6분)

**무엇을** · 두뇌가 코스를 걷는 동안 0.5초마다, 두뇌가 본 숫자를 사람이 읽는 글로 바꿉니다 (목표까지 1.84 m · 왼쪽 21° · 거리 센서 정면 0.84 m …).
보기는 **직진 · 왼쪽으로 · 오른쪽으로 · 제자리 좌회전 · 제자리 우회전** (물건 고르기는 머그컵 · 사과 · 랜턴 · 화분)이고, **정답은 내 두뇌의 답**입니다.
길을 조금 벗어난 상황도 배우도록 가끔(20%) 두뇌 대신 아무 보기로 걸어 봅니다. 그때도 정답은 두뇌의 답입니다.

**점수** · 강화학습에서 씁니다. 보기마다 '그렇게 하고 나머지 10초는 두뇌가 걸었으면 받은 보상'(흔들림 2번 평균 · 웹과 같은 공식 보상)입니다. 두뇌의 답보다 1점 넘게 낫거나 나쁜 보기만 반영합니다.

**정답률 상한** · 글에 적힌 숫자는 반올림되어 있어서, 두뇌에게 그 반올림한 숫자를 다시 줘도 같은 답을 내는 비율이 모델이 닿을 수 있는 정답률의 상한입니다.

✅ **정상이면** · 20판이면 학습 약 600~1,500문제 (과제마다 다름) · 정답률 상한 99% 안팎

In [ ]:
from connect_tycoon.llm import data as D
TRAIN_EPISODES = 20                                    # 늘릴수록 문제가 많아지고 지도학습이 길어집니다 (한 판 ≈ 30~80문제)
splits = {"train": range(10000, 10000 + TRAIN_EPISODES), "valid": range(20000, 20004), "test": range(30000, 30006)}
print(Q.build(brain, "link01_data", COURSE, splits=splits, log=None))
import json
meta = json.load(open("link01_data/meta.json"))
rows = D.load("link01_data", "train")
ex = rows[len(rows) // 3]
print(); print(ex["task"]["state"]); print(ex["task"]["question"])
for o, s in zip(ex["task"]["options"], ex["scores"]): print(f"  {o['label']} {o['description']:28s} 점수 {s:6.2f}")
print("정답(두뇌의 답)", ex["answer"], "· 정답률 상한", f"{meta['text_ceiling']['test']:.1%}")

## 3 · 학습 전 · 언어 모델을 로봇의 결정 자리에 앉혀 보기 (약 2~4분)

**무엇을** · 아무것도 배우지 않은 언어 모델에게 시험 문제를 풀게 하고(두뇌와 같은 답을 고르는 비율), 같은 코스를 20판 걷게 합니다 (씨앗 91001–91020 · 웹 공식 시험과 같은 씨앗).
**왜** · 학습 효과를 보려면 출발점을 먼저 재야 합니다.

✅ **정상이면** · 정답률 낮음 · 성공률이 두뇌보다 낮음

In [ ]:
from connect_tycoon.llm import train_torch as T
test = D.load("link01_data", "test")
results = {"두뇌 (선생님)": {"accuracy": 1.0, **{f"drive_{k}": v for k, v in mine.items() if k in ("success", "collide", "wrong", "ret", "time_s")}, "drive_runs": mine["n"]}}
base = T.Scorer()
results["학습 전"] = D.agreement(test, base.pick)
results["학습 전"].update(Q.drive_report(base.pick, brain, COURSE, log=None))
print(results["학습 전"])
del base; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [ ]:
import json
# 현재까지 기록된 성능(results) 변수 출력
print(json.dumps(results, indent=2, ensure_ascii=False))

### 🔍 학습 단계별 의사결정(Decision Making) 분석 코드
각 모델이 동일한 상황(지문)을 입력받았을 때 어떤 선택지(직진, 좌회전 등)에 대해 높은 확률(Probability)을 부여하는지 정량적으로 분석합니다.

In [18]:
import torch
import numpy as np
from connect_tycoon.llm import train_torch as T

# 분석에 사용할 하나의 예시 문제 가져오기
sample_case = test[0]
question_text = sample_case['task']['question']
state_text = sample_case['task']['state']
options = sample_case['task']['options']
correct_answer = sample_case['answer']

print("=== [분석용 시나리오 상황] ===")
print(state_text)
print(question_text)
for o in options:
    print(f"  {o['label']}: {o['description']}")
print(f"* 실제 두뇌(정답)의 결정: {correct_answer}\n")

# 각 단계별 가중치/어댑터 경로 지정
paths = {
    "1. 학습 전 (Base)": None,
    "2. 지도학습 (SFT)": "adapters/sft",
    "3. 강화학습 (RL)": "adapters/rl"
}

for name, adapter_path in paths.items():
    try:
        # 해당 가중치 로드하여 점수 평가기 인스턴스화
        scorer = T.Scorer(adapter_path) if adapter_path else T.Scorer()

        # 각 보기 알파벳(A, B, C, D, E)의 로그 확률(Log Likelihood) 계산
        choices = [o['label'] for o in options]
        scores = scorer.score_choices(state_text + "\n" + question_text, choices)

        # Softmax를 사용하여 로그 확률을 일반적인 선택 확률(0~1)로 변환
        exp_scores = np.exp(scores - np.max(scores))
        probabilities = exp_scores / np.sum(exp_scores)

        print(f"=== {name} 모델의 의사결정 확률 분포 ===")
        best_idx = np.argmax(probabilities)
        for i, (choice, prob) in enumerate(zip(choices, probabilities)):
            mark = "★" if choice == correct_answer else " "
            chk = "[선택]" if i == best_idx else "      "
            print(f"  {mark} {choice}: {prob:6.1%} {chk} ({options[i]['description']})")
        print()

        del scorer
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"⚠ {name} 모델 가중치를 불러올 수 없거나 아직 학습되지 않았습니다. ({e})\n")

=== [분석용 시나리오 상황] ===
LINK-01 로봇 · 장애물 피하기 · 장애물에 부딪히지 않고 화분(목표)까지 · 0.5초마다 한 번 고릅니다 (왼쪽 + · 오른쪽 −)
목표(화분)까지 3.47 m · 오른쪽 16°
거리 센서 (1.2 m 안에 막힌 곳까지 거리) · 오른쪽 100° 트임 · 오른쪽 80° 트임 · 오른쪽 60° 트임 · 오른쪽 40° 트임 · 오른쪽 20° 0.97 m · 정면 트임 · 왼쪽 20° 트임 · 왼쪽 40° 트임 · 왼쪽 60° 0.97 m · 왼쪽 80° 트임 · 왼쪽 100° 트임
지금 움직임 · 앞으로 0.00 m/초 · 회전 없음
직전 선택 없음 (첫 결정)
보상 · 가까워진 1 m마다 +5 · 도착 +10 · 부딪힘 −2 · 0.5초마다 −0.05
LINK-01이 다음 0.5초 동안 어떻게 걸을까요?
  A: 직진 · 앞으로 걷기
  B: 왼쪽으로 · 걸으면서 왼쪽으로 꺾기
  C: 오른쪽으로 · 걸으면서 오른쪽으로 꺾기
  D: 제자리 좌회전 · 멈춰서 왼쪽으로 돌기
  E: 제자리 우회전 · 멈춰서 오른쪽으로 돌기
* 실제 두뇌(정답)의 결정: A



Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

⚠ 1. 학습 전 (Base) 모델 가중치를 불러올 수 없거나 아직 학습되지 않았습니다. ('Scorer' object has no attribute 'score_choices')

⚠ 2. 지도학습 (SFT) 모델 가중치를 불러올 수 없거나 아직 학습되지 않았습니다. (adapters/sft is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`)

⚠ 3. 강화학습 (RL) 모델 가중치를 불러올 수 없거나 아직 학습되지 않았습니다. (adapters/rl is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`)



## 4 · 지도학습 · 내 두뇌의 답을 따라 배우기 (T4 예상 약 20~30분)

**무엇을** · 두뇌가 고른 보기 글자를 맞히도록 LoRA를 학습합니다 (모델 전체가 아니라 작은 덧붙임 층만).
**왜** · 두뇌(작은 신경망)가 숫자로 하던 판단을, 언어 모델이 글을 읽고 하도록 옮깁니다.
**드문 답은 여러 번** · 두뇌가 대부분 '직진'을 고르면, 그대로 배운 모델은 언제 꺾을지 배우기 전에 늘 직진만 고릅니다. 그래서 드문 답을 몇 번 반복해 보여 줍니다.

✅ **정상이면** · 정답률이 크게 오르고, 같은 코스 성공률이 두뇌 근처

In [ ]:
sft = T.sft("link01_data", "adapters/sft", epochs=2, lr=1e-4, batch=8)
results["지도학습"] = D.agreement(test, sft.pick)
results["지도학습"].update(Q.drive_report(sft.pick, brain, COURSE, log=None))
print(results["지도학습"])
del sft; torch.cuda.empty_cache()

## 5 · 강화학습 · 걸어 본 보상으로 고치기 (T4 예상 약 10~15분)

**무엇을** · 두뇌의 답보다 확실히(1점 넘게) 나은 보기의 확률은 올리고, 확실히 나쁜 보기(예: 장애물 쪽으로 꺾기)는 내립니다. 지도학습 모델에서 너무 멀어지지 않게 붙잡아 둡니다(KL).
**손실** · −Σ p(보기) · 장점(보기) + β · KL(p ‖ 지도학습 모델)

✅ **정상이면** · 성공률이 지도학습과 비슷하거나 오릅니다. 마지막 셀은 **같은 코스 20판 성공률이 더 높은 쪽**(같으면 정답률이 높은 쪽)을 올립니다

In [ ]:
rlm, hist = T.rl("link01_data", "adapters/sft", "adapters/rl", steps=100, batch=8, lr=5e-6, beta=0.2, max_rows=1500)
results["강화학습"] = D.agreement(test, rlm.pick)
results["강화학습"].update(Q.drive_report(rlm.pick, brain, COURSE, log=None))
print(f"{'':14s} 정답률  같은 코스 20판 성공  부딪힌 판"
for k, v in results.items(): print(f"{k:14s} {v['accuracy']:6.1%}  {v['drive_success']:8.0%}  {v['drive_collide']:12.0%}")
key = lambda k: (results[k]["drive_success"], results[k]["accuracy"])
BEST = "adapters/rl" if key("강화학습") >= key("지도학습") else "adapters/sft"
print("올릴 모델 ·", "강화학습" if BEST.endswith("rl") else "지도학습", "(같은 코스 성공률이 더 높은 쪽)")

## 6 · Hugging Face에 올리기

1. https://huggingface.co/settings/tokens 에서 **Write** 토큰을 만듭니다.
2. Colab 왼쪽 🔑(보안 비밀)에 이름 `HF_TOKEN`, 값에 토큰을 넣고 **노트북 액세스**를 켭니다.
3. **이름**을 영문으로 적습니다. 업로드 이름은 `connectailab-rl-이름-버전-link01` 입니다 (예: `connectailab-rl-jay-v1-link01`).
4. **나를 알리는 글**을 자유롭게 적으면 모델 카드 맨 위 '만든 사람'에 들어갑니다.

토큰은 코드에 적지 않습니다.

In [21]:
from google.colab import userdata
from connect_tycoon.link01.hub import push_decision_model
from connect_tycoon.llm.hub import make_author
import os

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
assert HF_TOKEN, "왼쪽 🔑에 HF_TOKEN(Write 권한)을 넣고 노트북 액세스를 켜 주세요"

NAME = "tazza11"                                         #@param {type:"string"}
VERSION = "v3"                                       #@param {type:"string"}
PRIVATE = False                                      #@param {type:"boolean"}

# 나를 알리는 글 · 형식 없이 자유롭게
ABOUT_ME = '''
'''
author = make_author(ABOUT_ME)

# 학습을 수행하지 않고 다이렉트로 올릴 때를 대비한 안전 장치 (Fallback)
if 'BEST' not in globals():
    print("⚠ 아직 지도학습/강화학습이 진행되지 않아 기본 디렉토리('adapters/sft')를 대상으로 업로드를 시도합니다.")
    if not os.path.exists("adapters/sft"):
        os.makedirs("adapters/sft", exist_ok=True)
    BEST = "adapters/sft"

if 'results' not in globals():
    results = {"두뇌 (선생님)": {"accuracy": 1.0, "drive_success": 1.0, "drive_collide": 0.0, "drive_wrong": 0.0}}

meta["chosen"] = "강화학습" if BEST.endswith("rl") else "지도학습"
url = push_decision_model(BEST, name=NAME, version=VERSION, results=results, meta=meta, token=HF_TOKEN, private=PRIVATE, author=author)
print("올라갔습니다 ·", url)

⚠ 아직 지도학습/강화학습이 진행되지 않아 기본 디렉토리('adapters/sft')를 대상으로 업로드를 시도합니다.


No files have been modified since last commit. Skipping to prevent empty commit.


올라갔습니다 · https://huggingface.co/tazza11/connectailab-rl-tazza11-v2-link01
